# 🏋️ Step 2 · Fine-tune YOLOv8-Face

**Training pipeline.** Fine-tune the pretrained face detector by running the `train_yolo` platform job — the same job that drift-triggered retraining reuses in notebook 6 — and inspect the registered model.

`📦 register train_yolo → 🚀 run job → 🏷️ model registry → 📥 inspect`

> ⚙️ The job requests **1 GPU** on the cluster; the notebook itself needs none.
> ℹ️ This folder must be inside the project filesystem (e.g. cloned into the `Jupyter` dataset) — it becomes the job's `appPath`.

In [ ]:
import os

import hopsworks
import run_job

project = hopsworks.login()
mr = project.get_model_registry()
print(f"✅ Connected to project: {project.name}")

### 📦 Register the training job
`run_job.ensure_train_yolo_job` is idempotent: it reuses the job if it exists, otherwise it locates this folder in the project filesystem and registers `train_yolo` (`train.py`, `yolov8` GPU environment).

In [ ]:
train_yolo_job = run_job.ensure_train_yolo_job(project)
train_yolo_job

### 🚀 Train
Runs remotely on a GPU node in default mode (fine-tune from the prepared snapshot in `data/widerface.zip`) and streams the job logs when it finishes. The job itself evaluates the model and registers it as `facerecognition`.

In [ ]:
execution = run_job.run_and_print_logs(train_yolo_job)
assert execution.success, "train_yolo failed — see the logs above"

### 📥 Inspect the registered model
Metrics and a face-detection preview on a sample image come from the job's artifacts.

In [ ]:
from IPython.display import Image as IPImage, display

models = mr.get_models("facerecognition")
if not models:
    raise RuntimeError("No 'facerecognition' model registered — did the job succeed?")
faces_model = max(models, key=lambda m: m.version)

print(f"✅ facerecognition v{faces_model.version}: {faces_model.training_metrics}")

local_dir = faces_model.download()
display(IPImage(os.path.join(local_dir, "images", "bus-faces-detected.png")))

### 🏆 Compare versions
Every run of `train_yolo` (including drift-triggered retrains from notebook 6) registers a new `facerecognition` version with mAP metrics, so versions are directly comparable.

In [ ]:
best_model = mr.get_best_model("facerecognition", "mAP50", "max")
print(f"🏆 Best facerecognition so far: v{best_model.version} ({best_model.training_metrics})")